<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=300>
 
# **Algoritmos y Programación III**
**Proyecto final — Clasificación de calidad de frutas**
 
---
 
- **Martinez Vasquez Luna Catalina — A00401964**
- **Mosquera Daza Renzo Fernando — A00401681**
- **Tobar Gómez Valentina — A00401749**
 
---
 
# Notebook 04 — Modelo de Deep Learning: Red Neuronal Convolucional (CNN)

## 1. Propósito
 
El propósito de este notebook es desarrollar y evaluar un modelo de aprendizaje profundo (CNN). **Importante:** a diferencia de los modelos tradicionales entrenados previamente sobre vectores de píxeles aplanados, las CNN aprovechan la estructura espacial de las imágenes para aprender automáticamente características visuales relevantes como bordes, texturas, manchas y formas permitiendo una representación más adecuada del contenido visual y un mejor desempeño en tareas de clasificación.

**Validación cruzada:** Para seleccionar la mejor configuración del modelo, se entrenaron varias variantes de la arquitectura modificando hiperparámetros como el número de filtros, la tasa de dropout y la tasa de aprendizaje. Cada variante se evaluó sobre un conjunto de validación fijo y la de mejor desempeño fue utilizada para la evaluación final en el conjunto de prueba.

## 2. Importación de librerías

Se cargan las librerías para entrenar el modelo, ajustar hiperparámetros y guardar el modelo que usará la aplicación.

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
 
import joblib
import tensorflow as tf
from tensorflow.keras import models, layers, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score, cohen_kappa_score
 
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 3. Rutas y parámetros
 
Se usan exactamente los mismos archivos que produjo el Notebook 02,
por lo que la partición train/val/test es idéntica a la de los modelos ML.
Esto hace que la comparación final de los cuatro modelos sea justa:
todos se evalúan sobre el mismo test set.
 
El tamaño de imagen se fija en **128×128**, el mismo que definió el Notebook 02
cuando guardó las imágenes procesadas. Los modelos ML del Notebook 03 usaron 48×48
para reducir el costo computacional del SVM; la CNN sí puede aprovechar la resolución
completa porque las convoluciones son eficientes en espacio.

In [ ]:
PROJECT_ROOT   = Path.cwd().parent
ANNOTATIONS_DIR = PROJECT_ROOT / "data" / "annotations"
MODELS_DIR     = PROJECT_ROOT / "models"
FIGURES_DIR    = PROJECT_ROOT / "results" / "figures"
TABLES_DIR     = PROJECT_ROOT / "results" / "tables"
 
TRAIN_PATH = ANNOTATIONS_DIR / "train_metadata.csv"
VAL_PATH   = ANNOTATIONS_DIR / "val_metadata.csv"
TEST_PATH  = ANNOTATIONS_DIR / "test_metadata.csv"
 
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS     = 40
QUALITY_LABELS = ["bad", "regular", "good"]
SIZE_LABELS    = ["small", "medium", "large"]
 
for directory in [MODELS_DIR, FIGURES_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
 
print("Proyecto:", PROJECT_ROOT)
print(f"Imagen de entrada: {IMG_SIZE} x {IMG_SIZE} x 3")
print(f"Batch size: {BATCH_SIZE} | Épocas máximas: {EPOCHS}")

## 4. Carga de metadatos y construcción de arrays
 
Cargamos los CSV de metadatos del Notebook 02 y leemos cada imagen
desde `processed_relative_path`. El preprocesamiento es idéntico al
que verá la aplicación en producción:
 
1. Abrir en RGB con corrección EXIF.
2. Redimensionar con padding blanco a 128×128 (`ImageOps.pad`),
   preservando el aspect ratio de la fruta.
3. Normalizar al rango **[0, 1]** dividiendo entre 255.

In [ ]:
def read_metadata(path, split_name):
    if not path.exists():
        raise FileNotFoundError(
            f"No existe {path}. Ejecuta primero el Notebook 02."
        )
    df = pd.read_csv(path)
    df["split"] = split_name
    return df
 
 
def load_images(df, img_size=IMG_SIZE):
    """
    Lee las imágenes listadas en df["processed_relative_path"],
    las redimensiona a img_size x img_size con padding blanco y
    devuelve un array float32 en [0, 1] de forma (N, H, W, 3).
    """
    imgs = []
    for rel_path in df["processed_relative_path"]:
        abs_path = PROJECT_ROOT / Path(str(rel_path).replace("\\\\", "/").strip())
        with Image.open(abs_path) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")
            img = ImageOps.pad(
                img,
                (img_size, img_size),
                color=(255, 255, 255),
                centering=(0.5, 0.5),
            )
        imgs.append(np.asarray(img, dtype="float32"))
    return np.stack(imgs) / 255.0
 
 
train_df = read_metadata(TRAIN_PATH, "train")
val_df   = read_metadata(VAL_PATH,   "val")
test_df  = read_metadata(TEST_PATH,  "test")
 
print("Cargando imágenes de entrenamiento…")
X_train = load_images(train_df)
print("Cargando imágenes de validación…")
X_val   = load_images(val_df)
print("Cargando imágenes de prueba…")
X_test  = load_images(test_df)
 
print(f"\\nX_train : {X_train.shape}  dtype={X_train.dtype}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Rango de valores: [{X_train.min():.2f}, {X_train.max():.2f}]")

In [ ]:
# Etiquetas
y_train_quality = train_df["quality_label"].astype(str).values
y_val_quality   = val_df["quality_label"].astype(str).values
y_test_quality  = test_df["quality_label"].astype(str).values
 
y_train_size = train_df["size_label"].astype(str).values
y_val_size   = val_df["size_label"].astype(str).values
y_test_size  = test_df["size_label"].astype(str).values
 
# Convertimos las etiquetas a enteros usando el orden fijo de las listas.
quality_to_int = {c: i for i, c in enumerate(QUALITY_LABELS)}
size_to_int    = {c: i for i, c in enumerate(SIZE_LABELS)}
 
y_train_q_int = np.array([quality_to_int[l] for l in y_train_quality])
y_val_q_int   = np.array([quality_to_int[l] for l in y_val_quality])
y_test_q_int  = np.array([quality_to_int[l] for l in y_test_quality])
 
y_train_s_int = np.array([size_to_int[l] for l in y_train_size])
y_val_s_int   = np.array([size_to_int[l] for l in y_val_size])
y_test_s_int  = np.array([size_to_int[l] for l in y_test_size])
 
print("Distribución quality — train:")
for cls, n in zip(QUALITY_LABELS,
                  np.bincount(y_train_q_int, minlength=3)):
    print(f"  {cls}: {n}")
 
print("\\nDistribución size — train:")
for cls, n in zip(SIZE_LABELS,
                  np.bincount(y_train_s_int, minlength=3)):
    print(f"  {cls}: {n}")

## 5. Arquitectura de la CNN
 
### Fundamento matemático
 
La operación central de la CNN es la **convolución discreta 2D**:
 
$$
(I * K)(i,j) = \sum_{m} \sum_{n} I(i+m,\; j+n) \cdot K(m,n)
$$
 
Aquí $I$ es la imagen (o el mapa de características de la capa anterior)
y $K$ es el filtro aprendido. En cada posición $(i,j)$ el filtro produce
un único valor que resume la presencia del patrón $K$ en esa vecindad.
 
La red aprende muchos filtros simultáneamente (32, 64, 128),
cada uno sensible a un patrón diferente donde algunos detectarán bordes,
otros manchas oscuras, otros texturas rugosas.
 
### Estructura de la CNN
 
La arquitectura implementada está compuesta por tres bloques convolucionales seguidos de una etapa de clasificación. Cada bloque combina una capa convolucional para la extracción de características, una capa de normalización (Batch Normalization) para estabilizar el entrenamiento y una capa de agrupamiento (Max Pooling) para reducir progresivamente la dimensionalidad espacial de la imagen.

| Etapa        | Operación                       | Dimensión de salida |
| ------------ | ------------------------------- | ------------------- |
| Entrada      | Imagen RGB                      | 128 × 128 × 3       |
| Bloque 1     | Conv2D (32 filtros, 3×3, ReLU)  | 128 × 128 × 32      |
|              | Batch Normalization             | 128 × 128 × 32      |
|              | MaxPooling2D (2×2)              | 64 × 64 × 32        |
| Bloque 2     | Conv2D (64 filtros, 3×3, ReLU)  | 64 × 64 × 64        |
|              | Batch Normalization             | 64 × 64 × 64        |
|              | MaxPooling2D (2×2)              | 32 × 32 × 64        |
| Bloque 3     | Conv2D (128 filtros, 3×3, ReLU) | 32 × 32 × 128       |
|              | Batch Normalization             | 32 × 32 × 128       |
|              | MaxPooling2D (2×2)              | 16 × 16 × 128       |
| Clasificador | Flatten                         | 32 768              |
|              | Dense (128 neuronas, ReLU)      | 128                 |
|              | Dropout (0.5)                   | 128                 |
| Salida       | Dense (3 neuronas, Softmax)     | 3 probabilidades    |

 
* **Batch Normalization:** estabiliza las activaciones de la red durante el entrenamiento, favoreciendo una convergencia más rápida y estable.
* **Dropout (0.5):** desactiva aleatoriamente una parte de las neuronas durante el entrenamiento para reducir el riesgo de sobreajuste y mejorar la capacidad de generalización.
* **Capa de salida Softmax:** se utiliza una capa de tres neuronas con función de activación softmax para obtener una distribución de probabilidades sobre las tres clases posibles.
* **Función de pérdida:** se emplea sparse_categorical_crossentropy, adecuada para problemas de clasificación multiclase cuando las etiquetas están codificadas como enteros.

In [ ]:
def build_model(num_classes, filters=(32, 64, 128),
                dense_units=128, dropout_rate=0.5,
                learning_rate=1e-3):
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
 
    x = layers.Conv2D(filters[0], (3, 3), activation="relu",
                      padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)        
 
    x = layers.Conv2D(filters[1], (3, 3), activation="relu",
                      padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)         
 
    x = layers.Conv2D(filters[2], (3, 3), activation="relu",
                      padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)         
 
    x = layers.Flatten()(x)                    
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
 
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model
 
model_demo = build_model(num_classes=3)
model_demo.summary()

## 6. Exploración de hiperparámetros
 
Con el fin de identificar la configuración más adecuada, se entrenaron 3 variantes de la arquitectura modificando hiperparámetros como el número de filtros, la tasa de dropout y la tasa de aprendizaje. Cada variante fue evaluada sobre el conjunto de validación y se seleccionó la de mejor desempeño (*val_accuracy*) para cada una de las tareas de clasificación.
 
| Variante | Filtros        | Dropout | Learning Rate | Descripción                       |
| -------- | -------------- | ------- | ------------- | --------------------------------- |
| A        | 32 / 64 / 128  | 0.5     | 1e-3          | Configuración base                |
| B        | 32 / 64 / 128  | 0.6     | 5e-4          | Mayor regularización              |
| C        | 64 / 128 / 256 | 0.5     | 1e-3          | Mayor capacidad de representación |

Hacemos esto **por separado para cada tarea** (quality y size),
porque son problemas distintos y podrían tener mejores hiperparámetros distintos.

In [ ]:
VARIANTS = {
    "A_base": dict(
        filters=(32, 64, 128),
        dense_units=128,
        dropout_rate=0.5,
        learning_rate=1e-3,
        description="Base",
    ),
    "B_mas_regularizacion": dict(
        filters=(32, 64, 128),
        dense_units=128,
        dropout_rate=0.6,
        learning_rate=5e-4,
        description="Más regularización",
    ),
    "C_mas_capacidad": dict(
        filters=(64, 128, 256),
        dense_units=256,
        dropout_rate=0.5,
        learning_rate=1e-3,
        description="Mayor capacidad de representación",
    ),
}
 
print("Variantes a entrenar:")
for name, cfg in VARIANTS.items():
    print(f"  {name}: {cfg['description']}")

Algunos de los callbacks implementados: 

* EarlyStopping: detiene el entrenamiento si val_loss no mejora en 8 épocas y restaura los mejores pesos, esto evita sobreajuste por entrenar demasiadas épocas.
* ReduceLROnPlateau: reduce la tasa de aprendizaje a la mitad si val_loss no mejora en 4 épocas, lo cual ayuda al modelo a converger más suave y preciso cerca del mínimo.

In [ ]:
def train_variant(X_tr, y_tr, X_v, y_v, variant_name, variant_cfg,
                  task_name, epochs=EPOCHS, batch_size=BATCH_SIZE):
    print(f"\\n{'='*60}")
    print(f"  Tarea: {task_name}  |  Variante: {variant_name}")
    print(f"  {variant_cfg['description']}")
    print(f"{'='*60}")
 
    cfg = {k: v for k, v in variant_cfg.items() if k != "description"}
    model = build_model(num_classes=len(np.unique(y_tr)), **cfg)
 
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),
    ]
 
    history = model.fit(
        X_tr, y_tr,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_v, y_v), 
        callbacks=callbacks,
        verbose=1,
    )
 
    best_val_acc = max(history.history["val_accuracy"])
    best_val_loss = min(history.history["val_loss"])
    print(f"\\n  → Mejor val_accuracy: {best_val_acc:.4f} | "
          f"Mejor val_loss: {best_val_loss:.4f}")
 
    return model, history

## 7. Entrenamiento para calidad (`quality_label`)
 
Entrenamos las tres variantes para la tarea de clasificar calidad
(bad / regular / good) y comparamos sus curvas de validación.

In [ ]:
quality_histories = {}
quality_models    = {}
 
for variant_name, variant_cfg in VARIANTS.items():
    model, history = train_variant(
        X_train, y_train_q_int,
        X_val,   y_val_q_int,
        variant_name=variant_name,
        variant_cfg=variant_cfg,
        task_name="quality",
    )
    quality_histories[variant_name] = history
    quality_models[variant_name]    = model

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
colors = {"A_base": "#2980b9",
          "B_mas_regularizacion": "#27ae60",
          "C_mas_capacidad": "#e74c3c"}
 
for name, hist in quality_histories.items():
    ax1.plot(hist.history["val_accuracy"], label=name,
             color=colors[name], linewidth=2)
    ax2.plot(hist.history["val_loss"],     label=name,
             color=colors[name], linewidth=2, linestyle="--")
 
ax1.set_title("val_accuracy por variante — quality", fontweight="bold")
ax1.set_xlabel("Época")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(alpha=0.3)
 
ax2.set_title("val_loss por variante — quality", fontweight="bold")
ax2.set_xlabel("Época")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_cnn_variantes_quality.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
variant_summary_quality = []
for name, hist in quality_histories.items():
    best_epoch = int(np.argmax(hist.history["val_accuracy"]))
    variant_summary_quality.append({
        "variante": name,
        "descripcion": VARIANTS[name]["description"],
        "epocas_entrenadas": len(hist.history["val_accuracy"]),
        "mejor_epoca": best_epoch + 1,
        "val_accuracy": round(max(hist.history["val_accuracy"]), 4),
        "val_loss":     round(min(hist.history["val_loss"]),     4),
    })
 
df_vs_quality = pd.DataFrame(variant_summary_quality).sort_values(
    "val_accuracy", ascending=False
)
display(df_vs_quality)
 
best_variant_quality = df_vs_quality.iloc[0]["variante"]
print(f"\\n→ Mejor variante para QUALITY: {best_variant_quality}")
print(f"  val_accuracy = {df_vs_quality.iloc[0]['val_accuracy']}")

## 8. Entrenamiento para tamaño (`size_label`)
 
Repetimos el mismo proceso para la tarea de clasificar tamaño
(small / medium / large). 

In [ ]:
size_histories = {}
size_models    = {}
 
for variant_name, variant_cfg in VARIANTS.items():
    model, history = train_variant(
        X_train, y_train_s_int,
        X_val,   y_val_s_int,
        variant_name=variant_name,
        variant_cfg=variant_cfg,
        task_name="size",
    )
    size_histories[variant_name] = history
    size_models[variant_name]    = model

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
 
for name, hist in size_histories.items():
    ax1.plot(hist.history["val_accuracy"], label=name,
             color=colors[name], linewidth=2)
    ax2.plot(hist.history["val_loss"],     label=name,
             color=colors[name], linewidth=2, linestyle="--")
 
ax1.set_title("val_accuracy por variante — size", fontweight="bold")
ax1.set_xlabel("Época")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(alpha=0.3)
 
ax2.set_title("val_loss por variante — size", fontweight="bold")
ax2.set_xlabel("Época")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_cnn_variantes_size.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
variant_summary_size = []
for name, hist in size_histories.items():
    best_epoch = int(np.argmax(hist.history["val_accuracy"]))
    variant_summary_size.append({
        "variante": name,
        "descripcion": VARIANTS[name]["description"],
        "epocas_entrenadas": len(hist.history["val_accuracy"]),
        "mejor_epoca": best_epoch + 1,
        "val_accuracy": round(max(hist.history["val_accuracy"]), 4),
        "val_loss":     round(min(hist.history["val_loss"]),     4),
    })
 
df_vs_size = pd.DataFrame(variant_summary_size).sort_values(
    "val_accuracy", ascending=False
)
display(df_vs_size)
 
best_variant_size = df_vs_size.iloc[0]["variante"]
print(f"\\n→ Mejor variante para SIZE: {best_variant_size}")
print(f"  val_accuracy = {df_vs_size.iloc[0]['val_accuracy']}")

## 9. Curvas de entrenamiento del mejor modelo
 
Graficamos las curvas completas (train vs. val) del mejor modelo
para cada tarea, con el fin de analizar si hay sobreajuste.

In [ ]:
def plot_training_curves(history, task_name, variant_name, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(
        f"Curvas de entrenamiento — {task_name} | {variant_name}",
        fontsize=13, fontweight="bold"
    )
 
    epocas = range(1, len(history.history["accuracy"]) + 1)
 
    ax1.plot(epocas, history.history["accuracy"],
             label="Entrenamiento", color="#2980b9", linewidth=2)
    ax1.plot(epocas, history.history["val_accuracy"],
             label="Validación", color="#e74c3c",
             linewidth=2, linestyle="--")
    ax1.set_title("Precisión (Accuracy)")
    ax1.set_xlabel("Época")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(alpha=0.3)
 
    ax2.plot(epocas, history.history["loss"],
             label="Entrenamiento", color="#2980b9", linewidth=2)
    ax2.plot(epocas, history.history["val_loss"],
             label="Validación", color="#e74c3c",
             linewidth=2, linestyle="--")
    ax2.set_title("Pérdida (Cross-Entropy Loss)")
    ax2.set_xlabel("Época")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(alpha=0.3)
 
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
 
    final_train = history.history["accuracy"][-1]
    best_val    = max(history.history["val_accuracy"])
    gap = final_train - best_val
    print(f"  Train accuracy final : {final_train:.4f}")
    print(f"  Mejor val accuracy   : {best_val:.4f}")
    print(f"  Gap (train - val)    : {gap:.4f}")
    if gap < 0.05:
        print("  Sin sobreajuste significativo (gap < 5%)")
    elif gap < 0.15:
        print(" Sobreajuste leve (gap 5-15%)")
    else:
        print("Sobreajuste detectado (gap > 15%)")
 
print("\\n── QUALITY ──────────────────────────────────────────────")
plot_training_curves(
    quality_histories[best_variant_quality],
    task_name="quality",
    variant_name=best_variant_quality,
    save_path=FIGURES_DIR / "04_curvas_quality.png",
)
 
print("\\n── SIZE ─────────────────────────────────────────────────")
plot_training_curves(
    size_histories[best_variant_size],
    task_name="size",
    variant_name=best_variant_size,
    save_path=FIGURES_DIR / "04_curvas_size.png",
)

## 10. Evaluación final en el conjunto de prueba
 
El conjunto de prueba no se tocó durante el entrenamiento ni la
selección de variantes, ya que es la evaluación definitiva del modelo.

In [ ]:
def evaluate_on_test(model, X, y_int, y_str, labels, model_name, task_name):
    y_pred_int = np.argmax(model.predict(X, verbose=0), axis=1)
    y_pred_str = np.array([labels[i] for i in y_pred_int])
 
    return {
        "target":          task_name,
        "dataset":         "test",
        "model":           model_name,
        "accuracy":        round(accuracy_score(y_str, y_pred_str), 4),
        "precision_macro": round(
            __import__("sklearn.metrics", fromlist=["precision_score"])
            .precision_score(y_str, y_pred_str, average="macro",
                             zero_division=0, labels=labels), 4),
        "recall_macro":    round(
            __import__("sklearn.metrics", fromlist=["recall_score"])
            .recall_score(y_str, y_pred_str, average="macro",
                          zero_division=0, labels=labels), 4),
        "f1_macro":        round(f1_score(
            y_str, y_pred_str, average="macro",
            zero_division=0, labels=labels), 4),
        "f1_weighted":     round(f1_score(
            y_str, y_pred_str, average="weighted",
            zero_division=0, labels=labels), 4),
        "kappa":           round(cohen_kappa_score(y_str, y_pred_str), 4),
        "labels":          ", ".join(labels),
    }
 
from sklearn.metrics import precision_score, recall_score
 
def evaluate_on_test(model, X, y_int, y_str, labels, model_name, task_name):
    y_pred_int = np.argmax(model.predict(X, verbose=0), axis=1)
    y_pred_str = np.array([labels[i] for i in y_pred_int])
    return {
        "target":          task_name,
        "dataset":         "test",
        "model":           model_name,
        "accuracy":        round(accuracy_score(y_str, y_pred_str), 4),
        "precision_macro": round(precision_score(
            y_str, y_pred_str, average="macro",
            zero_division=0, labels=labels), 4),
        "recall_macro":    round(recall_score(
            y_str, y_pred_str, average="macro",
            zero_division=0, labels=labels), 4),
        "f1_macro":        round(f1_score(
            y_str, y_pred_str, average="macro",
            zero_division=0, labels=labels), 4),
        "f1_weighted":     round(f1_score(
            y_str, y_pred_str, average="weighted",
            zero_division=0, labels=labels), 4),
        "kappa":           round(cohen_kappa_score(y_str, y_pred_str), 4),
        "labels":          ", ".join(labels),
    }
 
 
cnn_rows = []
 
best_cnn_quality = quality_models[best_variant_quality]
row_q = evaluate_on_test(
    best_cnn_quality, X_test,
    y_test_q_int, y_test_quality,
    QUALITY_LABELS, "CNN", "quality"
)
cnn_rows.append(row_q)
 
best_cnn_size = size_models[best_variant_size]
row_s = evaluate_on_test(
    best_cnn_size, X_test,
    y_test_s_int, y_test_size,
    SIZE_LABELS, "CNN", "size"
)
cnn_rows.append(row_s)
 
df_cnn_results = pd.DataFrame(cnn_rows)
display(df_cnn_results)

In [ ]:
for task_name, model, X, y_int, y_str, labels in [
    ("quality", best_cnn_quality,
     X_test, y_test_q_int, y_test_quality, QUALITY_LABELS),
    ("size",    best_cnn_size,
     X_test, y_test_s_int, y_test_size,    SIZE_LABELS),
]:
    y_pred_int = np.argmax(model.predict(X, verbose=0), axis=1)
    y_pred_str = np.array([labels[i] for i in y_pred_int])
 
    print(f"\\n{'─'*50}")
    print(f"  Reporte — tarea: {task_name.upper()}")
    print(f"  Variante: {best_variant_quality if task_name=='quality' else best_variant_size}")
    print(f"{'─'*50}")
    print(classification_report(y_str, y_pred_str,
                                 labels=labels, zero_division=0))

## 11. Matrices de confusión
 


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Matrices de confusión — CNN — conjunto de prueba",
             fontsize=13, fontweight="bold")
 
for ax, task_name, model, X, y_int, y_str, labels in [
    (axes[0], "quality", best_cnn_quality,
     X_test, y_test_q_int, y_test_quality, QUALITY_LABELS),
    (axes[1], "size",    best_cnn_size,
     X_test, y_test_s_int, y_test_size,    SIZE_LABELS),
]:
    y_pred_int = np.argmax(model.predict(X, verbose=0), axis=1)
    y_pred_str = np.array([labels[i] for i in y_pred_int])
 
    ConfusionMatrixDisplay.from_predictions(
        y_str, y_pred_str,
        labels=labels,
        normalize=None,
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"CNN — {task_name}", fontweight="bold")
 
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_confusion_cnn.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Visualización de mapas de características
 
Lo utilizamos para visualizar las características extraídas por la CNN y comprender qué patrones visuales son considerados por el modelo durante el proceso de clasificación.

In [ ]:
fig_imgs, axes_imgs = plt.subplots(1, 3, figsize=(9, 3))
fig_imgs.suptitle("Imágenes de prueba usadas para mapas de activación",
                  fontweight="bold")
 
sample_imgs = {}
for cls_idx, cls_name in enumerate(QUALITY_LABELS):
    mask = y_test_quality == cls_name
    if mask.sum() == 0:
        idx = 0
    else:
        idx = np.where(mask)[0][0]
    sample_imgs[cls_name] = X_test[idx]
    axes_imgs[cls_idx].imshow(X_test[idx])
    axes_imgs[cls_idx].set_title(cls_name)
    axes_imgs[cls_idx].axis("off")
 
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_muestras_activacion.png", dpi=120,
            bbox_inches="tight")
plt.show()

In [ ]:
conv_layer_names = [l.name for l in best_cnn_quality.layers
                    if "conv2d" in l.name]
print("Capas convolucionales disponibles:")
for i, name in enumerate(conv_layer_names):
    print(f"  {i}: {name}")
 
layers_to_show = [conv_layer_names[0], conv_layer_names[-1]]
 
for layer_name in layers_to_show:
    layer_obj  = best_cnn_quality.get_layer(layer_name)
    act_model  = Model(inputs=best_cnn_quality.input,
                       outputs=layer_obj.output)
 
    test_img = sample_imgs[QUALITY_LABELS[len(QUALITY_LABELS)//2]]
    img_batch = np.expand_dims(test_img, axis=0)
    activations = act_model.predict(img_batch, verbose=0)
 
    n_filters = activations.shape[-1]
    n_show    = min(n_filters, 32)
    cols = 8
    rows = int(np.ceil(n_show / cols))
 
    fig, axes = plt.subplots(rows, cols,
                             figsize=(16, 2 * rows))
    fig.suptitle(
        f"Mapas de características — capa: {layer_name} "
        f"({n_filters} filtros)",
        fontsize=12, fontweight="bold"
    )
    for i in range(n_show):
        ax = axes.flat[i]
        ax.imshow(activations[0, :, :, i], cmap="viridis")
        ax.axis("off")
        ax.set_title(f"F{i+1}", fontsize=7)
    for j in range(n_show, rows * cols):
        axes.flat[j].axis("off")
 
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / f"04_activaciones_{layer_name}.png",
        dpi=100, bbox_inches="tight"
    )
    plt.show()
 
    print(f"  Capa {layer_name}: activaciones con forma {activations.shape[1:]}")

## 13. Guardar modelos

In [ ]:
def save_cnn(model, task_name, labels, variant_name):
    model_path  = MODELS_DIR / f"cnn_{task_name}.keras"
    config_path = MODELS_DIR / f"cnn_{task_name}_config.json"
 
    model.save(model_path)
 
    config = {
        "task":          task_name,
        "target":        task_name,
        "classes":       labels,
        "labels":        labels,
        "img_size":      [IMG_SIZE, IMG_SIZE],
        "image_size":    [IMG_SIZE, IMG_SIZE],
        "color_mode":    "RGB",
        "scale":         255.0,
        "flatten":       False,        
        "model_family":  "cnn",
        "variant":       variant_name,
        "preprocessing": "ImageOps.pad, RGB, 128x128, numpy array, scale /255",
    }
    config_path.write_text(
        json.dumps(config, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"  Modelo guardado : {model_path.relative_to(PROJECT_ROOT)}")
    print(f"  Config guardado : {config_path.relative_to(PROJECT_ROOT)}")
    return model_path, config_path
 
 
print("Guardando modelos CNN…")
save_cnn(best_cnn_quality, "quality", QUALITY_LABELS, best_variant_quality)
save_cnn(best_cnn_size,    "size",    SIZE_LABELS,    best_variant_size)

## 14. Comparación final: CNN vs. modelos de Machine Learning
 

In [ ]:
nb03_path = TABLES_DIR / "03_model_comparison.csv"
 
if nb03_path.exists():
    df_nb03 = pd.read_csv(nb03_path)
    df_nb03_test = df_nb03[df_nb03["dataset"] == "test"].copy()

    df_all = pd.concat([df_nb03_test, df_cnn_results], ignore_index=True)
else:
    print("Ejecuta primero el Notebook 03 para ver la comparación completa.")
    df_all = df_cnn_results.copy()
 
cols_show = ["target", "model", "accuracy", "f1_macro",
             "f1_weighted", "kappa"]
df_display = df_all[[c for c in cols_show if c in df_all.columns]].copy()
df_display = df_display.sort_values(["target", "f1_macro"], ascending=[True, False])
display(df_display)
 
full_comparison_path = TABLES_DIR / "04_comparacion_final.csv"
df_all.to_csv(full_comparison_path, index=False, encoding="utf-8")
print(f"\\nTabla guardada en: {full_comparison_path.relative_to(PROJECT_ROOT)}")

In [ ]:
if nb03_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Comparación f1_macro — todos los modelos — test set",
                 fontsize=13, fontweight="bold")
 
    model_colors = {
        "Baseline": "#95a5a6",
        "Random Forest":    "#2980b9",
        "SVM":       "#8e44ad",
        "CNN":              "#e74c3c",
    }
 
    for ax, task in zip(axes, ["quality", "size"]):
        subset = df_all[df_all["target"] == task].copy()
        order = ["Baseline mayoría", "Random Forest", "SVM lineal", "CNN"]
        subset = subset.set_index("model").reindex(
            [m for m in order if m in subset["model"].values]
        ).reset_index()
 
        bar_colors = [model_colors.get(m, "#bdc3c7") for m in subset["model"]]
        bars = ax.bar(subset["model"], subset["f1_macro"],
                      color=bar_colors, edgecolor="white",
                      linewidth=1.5, width=0.5)
 
        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.008,
                f"{bar.get_height():.3f}",
                ha="center", va="bottom",
                fontsize=9, fontweight="bold"
            )
 
        ax.set_title(f"Tarea: {task}", fontweight="bold")
        ax.set_ylabel("F1-macro")
        ax.set_ylim(0, 1.1)
        ax.set_xticklabels(subset["model"], rotation=12, ha="right")
        ax.grid(axis="y", alpha=0.3)
 
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "04_comparacion_final.png",
                dpi=150, bbox_inches="tight")
    plt.show()